In [1]:
!pip install -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.8 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: datasets
    Found existing installatio

In [2]:
# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModel
from transformers import TrainingArguments, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 2) Load Data
# ------------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/shuvo128/rocccc/IMDB_Cleaned (1).csv")

text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

# ------------------------------------------------------------
# 3) Split (40k / 5k / 5k)
# ------------------------------------------------------------
train_texts = texts[:40000]
train_labels = labels[:40000]

val_texts = texts[40000:45000]
val_labels = labels[40000:45000]

test_texts = texts[45000:50000]
test_labels = labels[45000:50000]

# ------------------------------------------------------------
# 4) TF-IDF (improved)
# ------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()

# ------------------------------------------------------------
# 5) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=256)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 6) Dataset
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf, labels):
        self.encodings = encodings
        self.tfidf = tfidf
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc, val_tfidf, val_labels)
test_dataset  = HybridDataset(test_enc, test_tfidf, test_labels)

# ------------------------------------------------------------
# 7) MULTI-HEAD CROSS-ATTENTION MODEL 🔥
# ------------------------------------------------------------
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels, num_heads=4):
        super().__init__()

        self.electra = AutoModel.from_pretrained(model_name)
        hidden_size = self.electra.config.hidden_size

        self.tfidf_proj = nn.Linear(tfidf_dim, hidden_size)

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(hidden_size, hidden_size)
        self.key   = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)

        self.out = nn.Linear(hidden_size, hidden_size)

        self.norm = nn.LayerNorm(hidden_size)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_labels)
        )

    def split_heads(self, x):
        B, H = x.size()
        x = x.view(B, self.num_heads, self.head_dim)
        return x

    def forward(self, input_ids, attention_mask, tfidf, labels=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]  # (B, H)

        tfidf_embed = self.tfidf_proj(tfidf)

        Q = self.split_heads(self.query(cls_output))
        K = self.split_heads(self.key(tfidf_embed))
        V = self.split_heads(self.value(tfidf_embed))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn, V)
        context = context.view(context.size(0), -1)

        fused = self.norm(cls_output + self.out(context))  # residual + norm

        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, labels)

        return {"loss": loss, "logits": logits}

# Initialize
model = MultiHeadCrossAttention(
    model_name="google/electra-base-discriminator",
    tfidf_dim=10000,
    num_labels=2
)

# ------------------------------------------------------------
# 8) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# ------------------------------------------------------------
# 9) Training (best setup)
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 10) Train
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 11) Evaluate
# ------------------------------------------------------------
print("\nValidation:")
print(trainer.evaluate())

print("\nTest:")
test_results = trainer.predict(test_dataset)

test_preds = np.argmax(test_results.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
acc = accuracy_score(test_labels, test_preds)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

...


Validation:


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.222818,0.325494,10000,0.942800,0.936628,0.948728,0.942639


{'eval_loss': 0.32549354434013367, 'eval_accuracy': 0.9428, 'eval_precision': 0.9366281387006775, 'eval_recall': 0.9487283003633428, 'eval_f1': 0.9426393902928198}

Test:


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'accuracy': 0.948, 'precision': 0.9476813317479191, 'recall': 0.9491861849940453, 'f1': 0.9484331614438715}
